In [5]:
from pathlib import Path

print("cwd:", Path.cwd())
print("/content/drive/MyDrive exists:", Path("/content/drive/MyDrive").exists())

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount('/content/drive')

proj = Path("/content/drive/MyDrive/green-llm-token-research")
print("\nproject folder exists:", proj.exists())
if proj.exists():
    for p in sorted((proj / "data/raw").glob("*.jsonl")):
        print(f"  {p.name}  {p.stat().st_size:,} bytes")

cwd: /content
/content/drive/MyDrive exists: False
Mounted at /content/drive

project folder exists: True
  runs.jsonl  1,079,684 bytes
  runs_qwen25_3b_colab.jsonl  1,383,175 bytes


In [6]:
import json
from collections import Counter, defaultdict
from pathlib import Path

EXPECTED_MODEL, EXPECTED_TEMP, EXPECTED_SEED, PAIRS = "qwen2.5:3b-instruct", 0, 42, 100

# Locate the result file regardless of the current working directory. In Colab
# "!cd" runs in a subshell and does not persist, so a relative path can resolve
# against the wrong directory.
REL = "data/raw/runs_qwen25_3b_colab.jsonl"
CANDIDATES = [
    Path("/content/drive/MyDrive/green-llm-token-research") / REL,
    Path.cwd() / REL,
    Path(REL),
]

RESULT = next((p for p in CANDIDATES if p.exists()), None)
if RESULT is None:
    print("Could not find the result file. Tried:")
    for p in CANDIDATES:
        print("  -", p)
    print(f"\nCurrent working directory: {Path.cwd()}")
    raise SystemExit(
        "Run  %cd /content/drive/MyDrive/green-llm-token-research  "
        "(the %cd magic, not !cd) and try again."
    )

records = [json.loads(l) for l in RESULT.read_text(encoding="utf-8").splitlines() if l.strip()]
failures = []


def check(label, ok, detail=""):
    print(f"[{'PASS' if ok else 'FAIL'}] {label}" + (f"  -> {detail}" if detail else ""))
    if not ok:
        failures.append(label)


print(f"File: {RESULT}  ({RESULT.stat().st_size:,} bytes)\n")

# step 14
check("Total records == 200", len(records) == PAIRS * 2, f"got {len(records)}")

# step 15
variants = Counter(r["variant"] for r in records)
check("100 baseline + 100 optimized",
      variants.get("baseline") == PAIRS and variants.get("optimized") == PAIRS, str(dict(variants)))

# step 16
counts = Counter((r["prompt_id"], r["variant"]) for r in records)
dupes = {k: v for k, v in counts.items() if v != 1}
check("No duplicate prompt/variant records", not dupes, str(dupes) if dupes else "")
check("200 unique prompt/variant keys", len(counts) == PAIRS * 2, f"got {len(counts)}")

# step 17
settings = Counter((r["model"], r["temperature"], r["seed"]) for r in records)
check("Correct model / temperature / seed, single combination",
      len(settings) == 1 and (EXPECTED_MODEL, EXPECTED_TEMP, EXPECTED_SEED) in settings,
      str(dict(settings)))

# step 18
bad = [(r.get("prompt_id"), r.get("variant")) for r in records
       if (r.get("input_tokens", 0) or 0) <= 0 or (r.get("output_tokens", 0) or 0) <= 0
       or (r.get("total_tokens", 0) or 0) <= 0 or (r.get("wall_time_s", 0) or 0) <= 0
       or not str(r.get("response_text", "")).strip()]
check("All records have valid tokens, time and response text", not bad, str(bad[:8]))

truncated = [(r["prompt_id"], r["variant"]) for r in records
             if r.get("done_reason") not in (None, "stop")]
if truncated:
    print(f"[WARN] {len(truncated)} record(s) not finished with done_reason='stop': {truncated[:8]}")
    print("       Truncated generations cap output tokens and understate the reduction.")

# step 19
pairs = defaultdict(dict)
for r in records:
    pairs[r["prompt_id"]][r["variant"]] = r
complete = {k: v for k, v in pairs.items() if "baseline" in v and "optimized" in v}


def tot(v, f):
    return sum(p[v][f] for p in complete.values())


def red(b, o):
    return (b - o) / b * 100


print("\n=== QWEN2.5 3B SUMMARY ===")
for label, field in (("Input-token reduction", "input_tokens"),
                     ("Output-token reduction", "output_tokens"),
                     ("Total-token reduction", "total_tokens"),
                     ("Wall-time reduction", "wall_time_s")):
    print(f"{label}: {red(tot('baseline', field), tot('optimized', field)):.2f}%")

print(f"Optimized used fewer total tokens in: "
      f"{sum(p['optimized']['total_tokens'] < p['baseline']['total_tokens'] for p in complete.values())}/{PAIRS} pairs")
print(f"Optimized was faster in: "
      f"{sum(p['optimized']['wall_time_s'] < p['baseline']['wall_time_s'] for p in complete.values())}/{PAIRS} pairs")

print(f"\nTotal GPU wall time: {sum(r['wall_time_s'] for r in records) / 60:.1f} min")
print("\n" + ("ALL CHECKS PASSED" if not failures else f"FAILED CHECKS: {failures}"))


File: /content/drive/MyDrive/green-llm-token-research/data/raw/runs_qwen25_3b_colab.jsonl  (1,383,175 bytes)

[PASS] Total records == 200  -> got 200
[PASS] 100 baseline + 100 optimized  -> {'baseline': 100, 'optimized': 100}
[PASS] No duplicate prompt/variant records
[PASS] 200 unique prompt/variant keys  -> got 200
[PASS] Correct model / temperature / seed, single combination  -> {('qwen2.5:3b-instruct', 0, 42): 200}
[PASS] All records have valid tokens, time and response text  -> []
[WARN] 4 record(s) not finished with done_reason='stop': [('P0026', 'baseline'), ('P0045', 'baseline'), ('P0045', 'optimized'), ('P0055', 'optimized')]
       Truncated generations cap output tokens and understate the reduction.

=== QWEN2.5 3B SUMMARY ===
Input-token reduction: 52.20%
Output-token reduction: 3.25%
Total-token reduction: 10.76%
Wall-time reduction: 2.11%
Optimized used fewer total tokens in: 77/100 pairs
Optimized was faster in: 66/100 pairs

Total GPU wall time: 90.5 min

ALL CHECKS PAS